In [1]:
import pandas as pd
import os
import glob

# --- Configuration ---
input_csv_dir = './data/'
output_parquet_dir = './data/parquet/'
csv_file_pattern = 'yellow_tripdata_*.csv'

# --- File Processing and Conversion to Parquet ---
# Create the output directory for Parquet files if it doesn't exist
if not os.path.exists(output_parquet_dir):
    os.makedirs(output_parquet_dir)
    print(f"Created directory: {output_parquet_dir}")

# Find all CSV files matching the pattern in the input directory
csv_files = glob.glob(os.path.join(input_csv_dir, csv_file_pattern))

if not csv_files:
    print(f"No CSV files found matching the pattern '{csv_file_pattern}' in '{input_csv_dir}'. Please check the directory and file names.")
else:
    print(f"Found {len(csv_files)} CSV files to process.")

    # Process each CSV file
    for csv_file_path in csv_files:
        # Construct the output Parquet file path
        csv_filename = os.path.basename(csv_file_path)
        parquet_filename = os.path.splitext(csv_filename)[0] + '.parquet'
        parquet_file_path = os.path.join(output_parquet_dir, parquet_filename)

        # Check if the Parquet file already exists to skip reprocessing
        if os.path.exists(parquet_file_path):
            print(f"Parquet file already exists, skipping conversion: {parquet_file_path}")
        else:
            print(f"Processing and converting to Parquet: {csv_file_path}")
            try:
                # Read the CSV file
                # Specify dtype=str for columns that might contain mixed types
                # Parse dates explicitly
                df_chunk = pd.read_csv(
                    csv_file_path,
                    parse_dates=['tpep_pickup_datetime', 'tpep_dropoff_datetime'],
                    # Add other dtype specifications if needed based on data inspection
                )

                # Save the DataFrame chunk to Parquet
                df_chunk.to_parquet(parquet_file_path, index=False) # index=False prevents writing the pandas index as a column
                print(f"Successfully converted to Parquet: {parquet_file_path}")

            except Exception as e:
                print(f"Error processing {csv_file_path}: {e}")

    # --- Read all Parquet files into a single DataFrame ---
    print(f"\nReading all Parquet files from '{output_parquet_dir}'...")
    try:
        # Read all Parquet files in the directory into one DataFrame
        # pandas.read_parquet can read from a directory containing multiple files
        df = pd.read_parquet(output_parquet_dir, engine='pyarrow')
        print(f"Successfully loaded combined data into DataFrame. Shape: {df.shape}")

        # --- Generate trip_id if it doesn't exist ---
        # Use the DataFrame index as the trip_id, ensuring it's a unique identifier for each row
        # This will generate unique IDs across all combined data
        if 'trip_id' not in df.columns:
            df['trip_id'] = df.index
            print("Generated 'trip_id' column from DataFrame index for combined data.")
        else:
            print("'trip_id' column already exists in the combined DataFrame.")

        # --- Create Dimension Tables ---

        # Datetime Dimension Table
        # Extract date and time components from pickup and dropoff datetimes
        datetime_dim = df[['tpep_pickup_datetime', 'tpep_dropoff_datetime']].drop_duplicates().reset_index(drop=True)
        datetime_dim['pickup_hour'] = datetime_dim['tpep_pickup_datetime'].dt.hour
        datetime_dim['pickup_day'] = datetime_dim['tpep_pickup_datetime'].dt.day
        datetime_dim['pickup_month'] = datetime_dim['tpep_pickup_datetime'].dt.month
        datetime_dim['pickup_year'] = datetime_dim['tpep_pickup_datetime'].dt.year
        datetime_dim['pickup_weekday'] = datetime_dim['tpep_pickup_datetime'].dt.weekday # Monday=0, Sunday=6
        datetime_dim['dropoff_hour'] = datetime_dim['tpep_dropoff_datetime'].dt.hour
        datetime_dim['dropoff_day'] = datetime_dim['tpep_dropoff_datetime'].dt.day
        datetime_dim['dropoff_month'] = datetime_dim['tpep_dropoff_datetime'].dt.month
        datetime_dim['dropoff_year'] = datetime_dim['tpep_dropoff_datetime'].dt.year
        datetime_dim['dropoff_weekday'] = datetime_dim['tpep_dropoff_datetime'].dt.weekday # Monday=0, Sunday=6

        # Add a unique primary key for the datetime dimension
        datetime_dim['datetime_id'] = datetime_dim.index

        # Reorder columns to match the schema definition
        datetime_dim = datetime_dim[['datetime_id', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
                                     'pickup_hour', 'pickup_day', 'pickup_month', 'pickup_year', 'pickup_weekday',
                                     'dropoff_hour', 'dropoff_day', 'dropoff_month', 'dropoff_year', 'dropoff_weekday']]


        # Passenger Count Dimension Table
        passenger_count_dim = df[['passenger_count']].drop_duplicates().reset_index(drop=True)
        # Add a unique primary key
        passenger_count_dim['passenger_count_id'] = passenger_count_dim.index
        # Reorder columns
        passenger_count_dim = passenger_count_dim[['passenger_count_id', 'passenger_count']]


        # Pickup Location Dimension Table
        # Note: Assuming pickup_latitude and pickup_longitude uniquely identify a pickup location
        pickup_location_dim = df[['pickup_latitude', 'pickup_longitude']].drop_duplicates().reset_index(drop=True)
        # Add a unique primary key
        pickup_location_dim['pickup_location_id'] = pickup_location_dim.index
        # Reorder columns
        pickup_location_dim = pickup_location_dim[['pickup_location_id', 'pickup_latitude', 'pickup_longitude']]


        # Dropoff Location Dimension Table
        # Note: Assuming dropoff_latitude and dropoff_longitude uniquely identify a dropoff location
        dropoff_location_dim = df[['dropoff_latitude', 'dropoff_longitude']].drop_duplicates().reset_index(drop=True)
        # Add a unique primary key
        dropoff_location_dim['dropoff_location_id'] = dropoff_location_dim.index
        # Reorder columns
        dropoff_location_dim = dropoff_location_dim[['dropoff_location_id', 'dropoff_latitude', 'dropoff_longitude']]


        # Trip Distance Dimension Table
        trip_distance_dim = df[['trip_distance']].drop_duplicates().reset_index(drop=True)
        # Add a unique primary key
        trip_distance_dim['trip_distance_id'] = trip_distance_dim.index
        # Reorder columns
        trip_distance_dim = trip_distance_dim[['trip_distance_id', 'trip_distance']]


        # Rate Code Dimension Table
        # Need to map RatecodeID to a name. Using a simple mapping based on common taxi rate codes.
        # You might need to adjust this mapping based on your data source's documentation.
        rate_code_mapping = {
            1: 'Standard rate',
            2: 'JFK',
            3: 'Newark',
            4: 'Nassau or Westchester',
            5: 'Negotiated fare',
            6: 'Group ride'
        }
        rate_code_dim = df[['RateCodeID']].drop_duplicates().reset_index(drop=True)
        rate_code_dim['rate_code_name'] = rate_code_dim['RateCodeID'].map(rate_code_mapping).fillna('Unknown') # Handle potential unknown codes
        # Add a unique primary key
        rate_code_dim['rate_code_id'] = rate_code_dim.index
        # Rename the RatecodeID column to rate_code to match schema
        rate_code_dim = rate_code_dim.rename(columns={'RateCodeID': 'rate_code'})
        # Reorder columns
        rate_code_dim = rate_code_dim[['rate_code_id', 'rate_code', 'rate_code_name']]


        # Payment Type Dimension Table
        # Need to map payment_type to a name. Using a simple mapping based on common taxi payment types.
        # You might need to adjust this mapping based on your data source's documentation.
        payment_type_mapping = {
            1: 'Credit card',
            2: 'Cash',
            3: 'No charge',
            4: 'Dispute',
            5: 'Unknown',
            6: 'Voided trip'
        }
        payment_type_dim = df[['payment_type']].drop_duplicates().reset_index(drop=True)
        payment_type_dim['payment_type_name'] = payment_type_dim['payment_type'].map(payment_type_mapping).fillna('Unknown') # Handle potential unknown types
        # Add a unique primary key
        payment_type_dim['payment_type_id'] = payment_type_dim.index
        # Reorder columns
        payment_type_dim = payment_type_dim[['payment_type_id', 'payment_type', 'payment_type_name']]


        # --- Create Fact Table ---

        # Start with the original dataframe and select relevant columns for the fact table
        # Now including the generated 'trip_id'
        fact_table = df[['trip_id', 'VendorID',
                         'tpep_pickup_datetime', 'tpep_dropoff_datetime',
                         'passenger_count', 'trip_distance',
                         'pickup_latitude', 'pickup_longitude',
                         'dropoff_latitude', 'dropoff_longitude',
                         'RateCodeID', 'payment_type',
                         'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
                         'improvement_surcharge', 'total_amount']].copy() # Use .copy() to avoid SettingWithCopyWarning

        # Join with dimension tables to get the dimension IDs

        # Join with datetime_dim
        fact_table = pd.merge(fact_table, datetime_dim, on=['tpep_pickup_datetime', 'tpep_dropoff_datetime'], how='left')

        # Join with passenger_count_dim
        fact_table = pd.merge(fact_table, passenger_count_dim, on='passenger_count', how='left')

        # Join with pickup_location_dim
        fact_table = pd.merge(fact_table, pickup_location_dim, on=['pickup_latitude', 'pickup_longitude'], how='left')

        # Join with dropoff_location_dim
        fact_table = pd.merge(fact_table, dropoff_location_dim, on=['dropoff_latitude', 'dropoff_longitude'], how='left')

        # Join with trip_distance_dim
        fact_table = pd.merge(fact_table, trip_distance_dim, on='trip_distance', how='left')

        # Join with rate_code_dim
        fact_table = pd.merge(fact_table, rate_code_dim[['rate_code_id', 'rate_code']], left_on='RateCodeID', right_on='rate_code', how='left') # Join on the actual code

        # Join with payment_type_dim
        fact_table = pd.merge(fact_table, payment_type_dim[['payment_type_id', 'payment_type']], on='payment_type', how='left') # Join on the actual type

        # Select and reorder columns for the final fact table
        fact_table = fact_table[['trip_id', 'VendorID', 'datetime_id', 'passenger_count_id',
                                 'trip_distance_id', 'pickup_location_id', 'dropoff_location_id',
                                 'rate_code_id', 'payment_type_id',
                                 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
                                 'improvement_surcharge', 'total_amount']]

        # Rename VendorID to vendor_id to match schema
        fact_table = fact_table.rename(columns={'VendorID': 'vendor_id'})
        print("\nStar schema tables successfully generated.")

        # Drop any duplicate rows in the fact table
        print(f"Fact table shape before dropping duplicates: {fact_table.shape}")
        fact_table = fact_table.drop_duplicates()
        print(f"Fact table shape after dropping duplicates: {fact_table.shape}")

    except Exception as e:
        print(f"\nError loading Parquet files or generating star schema: {e}")
        print("Ensure that the Parquet files were created successfully and that there is enough memory to load the combined data.")

Created directory: ./data/parquet/
Found 4 CSV files to process.
Processing and converting to Parquet: ./data\yellow_tripdata_2015-01.csv
Successfully converted to Parquet: ./data/parquet/yellow_tripdata_2015-01.parquet
Processing and converting to Parquet: ./data\yellow_tripdata_2016-01.csv
Successfully converted to Parquet: ./data/parquet/yellow_tripdata_2016-01.parquet
Processing and converting to Parquet: ./data\yellow_tripdata_2016-02.csv
Successfully converted to Parquet: ./data/parquet/yellow_tripdata_2016-02.parquet
Processing and converting to Parquet: ./data\yellow_tripdata_2016-03.csv
Successfully converted to Parquet: ./data/parquet/yellow_tripdata_2016-03.parquet

Reading all Parquet files from './data/parquet/'...
Successfully loaded combined data into DataFrame. Shape: (47248845, 19)
Generated 'trip_id' column from DataFrame index for combined data.

Star schema tables successfully generated.
Fact table shape before dropping duplicates: (47248845, 16)
Fact table shape af

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47248845 entries, 0 to 47248844
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int64         
 1   tpep_pickup_datetime   datetime64[ns]
 2   tpep_dropoff_datetime  datetime64[ns]
 3   passenger_count        int64         
 4   trip_distance          float64       
 5   pickup_longitude       float64       
 6   pickup_latitude        float64       
 7   RateCodeID             float64       
 8   store_and_fwd_flag     object        
 9   dropoff_longitude      float64       
 10  dropoff_latitude       float64       
 11  payment_type           int64         
 12  fare_amount            float64       
 13  extra                  float64       
 14  mta_tax                float64       
 15  tip_amount             float64       
 16  tolls_amount           float64       
 17  improvement_surcharge  float64       
 18  total_amount        

In [3]:
# Display the first few rows of each table to verify
print("\n--- Generated Tables ---")
print("Datetime Dimension Table:")
datetime_dim


--- Generated Tables ---
Datetime Dimension Table:


,datetime_id,tpep_pickup_datetime,tpep_dropoff_datetime,pickup_hour,pickup_day,pickup_month,pickup_year,pickup_weekday,dropoff_hour,dropoff_day,dropoff_month,dropoff_year,dropoff_weekday
0,0,2015-01-15 19:05:39,2015-01-15 19:23:42,19,15,1,2015,3,19,15,1,2015,3
1,1,2015-01-10 20:33:38,2015-01-10 20:53:28,20,10,1,2015,5,20,10,1,2015,5
2,2,2015-01-10 20:33:38,2015-01-10 20:43:41,20,10,1,2015,5,20,10,1,2015,5
3,3,2015-01-10 20:33:39,2015-01-10 20:35:31,20,10,1,2015,5,20,10,1,2015,5
4,4,2015-01-10 20:33:39,2015-01-10 20:52:58,20,10,1,2015,5,20,10,1,2015,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
47138628,47138628,2016-03-20 08:59:21,2016-04-18 10:58:05,8,20,3,2016,6,10,18,4,2016,0
47138629,47138629,2016-03-26 03:02:32,2016-06-14 18:47:55,3,26,3,2016,5,18,14,6,2016,1
47138630,47138630,2016-03-20 08:43:59,2016-06-27 15:05:01,8,20,3,2016,6,15,27,6,2016,0
47138631,47138631,2016-03-20 08:49:47,2016-06-28 19:11:27,8,20,3,2016,6,19,28,6,2016,1


In [4]:
print("\nPassenger Count Dimension Table:")
passenger_count_dim


Passenger Count Dimension Table:


,passenger_count_id,passenger_count
0,0,1
1,1,3
2,2,2
3,3,5
4,4,6
5,5,4
6,6,0
7,7,9
8,8,7
9,9,8


In [5]:
print("\nPickup Location Dimension Table:")
pickup_location_dim


Pickup Location Dimension Table:


,pickup_location_id,pickup_latitude,pickup_longitude
0,0,40.750111,-73.993896
1,1,40.724243,-74.001648
2,2,40.802788,-73.963341
3,3,40.713818,-74.009087
4,4,40.762428,-73.971176
...,...,...,...
23660078,23660078,40.641743,-73.789223
23660079,23660079,40.748898,-73.937790
23660080,23660080,40.644451,-73.790565
23660081,23660081,40.647377,-73.790077


In [6]:
print("\nDropoff Location Dimension Table:")
dropoff_location_dim


Dropoff Location Dimension Table:


,dropoff_location_id,dropoff_latitude,dropoff_longitude
0,0,40.750618,-73.974785
1,1,40.759109,-73.994415
2,2,40.824413,-73.951820
3,3,40.719986,-74.004326
4,4,40.742653,-74.004181
...,...,...,...
29129077,29129077,40.731781,-74.003151
29129078,29129078,40.757648,-73.875351
29129079,29129079,40.710987,-74.008614
29129080,29129080,40.578457,-73.971756


In [7]:
print("\nTrip Distance Dimension Table:")
trip_distance_dim


Trip Distance Dimension Table:


,trip_distance_id,trip_distance
0,0,1.59
1,1,3.30
2,2,1.80
3,3,0.50
4,4,3.00
...,...,...
5937,5937,161.40
5938,5938,46.28
5939,5939,44.02
5940,5940,54.88


In [8]:
print("\nRate Code Dimension Table:")
rate_code_dim


Rate Code Dimension Table:


,rate_code_id,rate_code,rate_code_name
0,0,1.0,Standard rate
1,1,2.0,JFK
2,2,5.0,Negotiated fare
3,3,3.0,Newark
4,4,4.0,Nassau or Westchester
5,5,99.0,Unknown
6,6,6.0,Group ride
7,7,NaN,Unknown


In [9]:
print("\nPayment Type Dimension Table:")
payment_type_dim


Payment Type Dimension Table:


,payment_type_id,payment_type,payment_type_name
0,0,1,Credit card
1,1,2,Cash
2,2,3,No charge
3,3,4,Dispute
4,4,5,Unknown


In [10]:
print("\nFact Table:")
fact_table


Fact Table:


,trip_id,vendor_id,datetime_id,passenger_count_id,trip_distance_id,pickup_location_id,dropoff_location_id,rate_code_id,payment_type_id,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
0,0,2,0,0,0,0,0,0,0,12.0,1.0,0.5,3.25,0.00,0.3,17.05
1,1,1,1,0,1,1,1,0,0,14.5,0.5,0.5,2.00,0.00,0.3,17.80
2,2,1,2,0,2,2,2,0,1,9.5,0.5,0.5,0.00,0.00,0.3,10.80
3,3,1,3,0,3,3,3,0,1,3.5,0.5,0.5,0.00,0.00,0.3,4.80
4,4,1,4,0,4,4,4,0,1,15.0,0.5,0.5,0.00,0.00,0.3,16.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47248840,47248840,1,46964686,0,98,31,31,7,1,19.0,1.0,0.5,0.00,0.00,0.3,20.80
47248841,47248841,1,47138629,0,112,9217022,237224,7,0,4.0,1.0,0.5,1.70,0.00,0.3,7.50
47248842,47248842,1,47138630,0,754,31,29129079,7,0,52.0,0.0,0.5,6.00,5.54,0.3,64.34
47248843,47248843,1,47138631,0,1339,23660081,29129080,7,0,42.5,1.0,0.5,5.00,0.00,0.3,49.30
